In [ ]:
import os
import pandas as pd
import kagglehub
path = kagglehub.dataset_download("silicon99/dft-accident-data")
print("Path to dataset files:", path)
data_path=os.path.join(path, "Accidents0515.csv")
df=pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
for column in df.columns:
    print(column)

In [ ]:
selected_columns = [
    'Accident_Index',
    'Longitude',
    'Latitude',
    'Accident_Severity',
    'Date',
    'Time',
    'Day_of_Week',
    'Number_of_Vehicles',
    'Number_of_Casualties',
    'Road_Type',
    'Speed_limit',
    'Junction_Detail',
    'Junction_Control',
    'Light_Conditions',
    'Weather_Conditions',
    'Road_Surface_Conditions',
    'Urban_or_Rural_Area'
]

In [ ]:
accidents_df = df[selected_columns]

In [ ]:
accidents_df.head()

In [ ]:
accidents_df.shape

In [ ]:
accidents_df['Date'] = pd.to_datetime(
    accidents_df['Date'],
    dayfirst=True,
    errors='coerce'
)

In [ ]:
accidents_df = accidents_df[
    accidents_df['Date'].dt.year.isin([2013, 2014, 2015])
].copy()

In [ ]:
accidents_df['Year'] = accidents_df['Date'].dt.year

In [ ]:
accidents_df['Year'].value_counts().sort_index()

In [ ]:
accidents_df.shape

In [ ]:
accidents_df.head()

In [ ]:
accidents_df.info()

In [ ]:
accidents_df.isnull().sum()

In [ ]:
missing_values = accidents_df.isnull().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

In [ ]:
accidents_df.duplicated().sum()

In [ ]:
accidents_df['Accident_Index'].duplicated().sum()

In [ ]:
accidents_df['Accident_Severity'].value_counts().sort_index()

In [ ]:
accidents_df['Accident_Severity'].value_counts(
    normalize=True
).sort_index() * 100

In [ ]:
accidents_df['Year'].value_counts().sort_index()

In [ ]:
accidents_df.describe()

In [ ]:
print("Before Cleaning:", accidents_df.shape)

In [ ]:
accidents_df = accidents_df.dropna(
    subset=['Longitude', 'Latitude', 'Time']
).copy()

In [ ]:
print("After Cleaning:", accidents_df.shape)

accidents_df[
    ['Longitude', 'Latitude', 'Time']
].isnull().sum()

In [ ]:
severity_names = {
    1: 'Fatal',
    2: 'Serious',
    3: 'Slight'
}

accidents_df['Severity_Label'] = accidents_df[
    'Accident_Severity'
].map(severity_names)

In [ ]:
accidents_df[
    ['Accident_Severity', 'Severity_Label']
].head()

In [ ]:
accidents_df['Severe_Accident'] = accidents_df[
    'Accident_Severity'
].isin([1, 2]).astype(int)

In [ ]:
accidents_df['Severe_Accident'].value_counts()

In [ ]:
accidents_df['Severe_Accident'].value_counts(
    normalize=True
) * 100

In [ ]:
accidents_df['Accident_DateTime'] = pd.to_datetime(
    accidents_df['Date'].astype(str)
    + ' '
    + accidents_df['Time'],
    errors='coerce'
)

In [ ]:
accidents_df['Month'] = accidents_df[
    'Accident_DateTime'
].dt.month

accidents_df['Hour'] = accidents_df[
    'Accident_DateTime'
].dt.hour

In [ ]:
accidents_df['Weather_Hour'] = accidents_df[
    'Accident_DateTime'
].dt.floor('h')

In [ ]:
accidents_df[
    [
        'Date',
        'Time',
        'Accident_DateTime',
        'Year',
        'Month',
        'Hour',
        'Weather_Hour'
    ]
].head()

In [ ]:
accidents_df = accidents_df.dropna(
    subset=['Longitude', 'Latitude', 'Time']
)

accidents_df.shape

In [ ]:
accidents_df[
    ['Longitude', 'Latitude', 'Time']
].isnull().sum()

In [ ]:
accidents_df[
    [
        'Accident_Severity',
        'Severity_Label',
        'Severe_Accident',
        'Accident_DateTime',
        'Month',
        'Hour',
        'Weather_Hour'
    ]
].head()

In [ ]:
accidents_df['Severe_Accident'].value_counts()

In [ ]:
accidents_df['Severe_Accident'].value_counts(normalize=True) * 100

In [ ]:
accidents_df['Accident_DateTime'].isnull().sum()

In [ ]:
accidents_df[
    ['Latitude', 'Longitude']
].describe()

In [ ]:
accidents_df['Grid_Latitude'] = accidents_df[
    'Latitude'
].round(1)

accidents_df['Grid_Longitude'] = accidents_df[
    'Longitude'
].round(1)

In [ ]:
accidents_df[
    ['Grid_Latitude', 'Grid_Longitude']
].drop_duplicates().shape

In [ ]:
police_df = df[['Accident_Index', 'Police_Force']]
police_df.head()

In [ ]:
accidents_df = accidents_df.merge(
    police_df,
    on='Accident_Index',
    how='left'
)

In [ ]:
accidents_df['Police_Force'].isnull().sum()

In [ ]:
accidents_df = accidents_df[
    accidents_df['Police_Force'].isin([1, 48])
].copy()

In [ ]:
accidents_df.shape

In [ ]:
accidents_df['Year'].value_counts().sort_index()

In [ ]:
accidents_df[
    ['Grid_Latitude', 'Grid_Longitude']
].drop_duplicates().shape

In [ ]:
accidents_df['Year'].value_counts().sort_index()

In [ ]:
accidents_df['Severity_Label'].value_counts()

In [ ]:
accidents_df['Severe_Accident'].value_counts(normalize=True) * 100

In [ ]:
accidents_df = accidents_df.reset_index(drop=True)

In [ ]:
accidents_df.head()

In [ ]:
os.makedirs("data", exist_ok=True)
accidents_df.to_csv("data/accidents_london_2013_2015.csv", index=False)

In [ ]:
print(accidents_df.shape)

In [ ]:
import requests

In [ ]:
test_latitude = accidents_df.loc[0, 'Grid_Latitude']
test_longitude = accidents_df.loc[0, 'Grid_Longitude']
test_date = accidents_df.loc[0, 'Date'].strftime('%Y-%m-%d')

print(test_latitude)
print(test_longitude)
print(test_date)

In [ ]:
weather_url = 'https://archive-api.open-meteo.com/v1/archive'

weather_params = {
    'latitude': test_latitude,
    'longitude': test_longitude,
    'start_date': test_date,
    'end_date': test_date,
    'hourly': [
        'temperature_2m',
        'rain',
        'snowfall',
        'weather_code',
        'cloud_cover',
        'wind_speed_10m'
    ],
    'timezone': 'Europe/London'
}

response = requests.get(
    weather_url,
    params=weather_params,
    timeout=60
)

print(response.status_code)

In [ ]:
weather_data = response.json()

weather_data.keys()

In [ ]:
weather_test_df = pd.DataFrame(
    weather_data['hourly']
)

weather_test_df.head()

In [ ]:
weather_test_df.shape

In [ ]:
weather_test_df.columns

In [ ]:
grid_df = accidents_df[
    ['Grid_Latitude', 'Grid_Longitude']
].drop_duplicates().reset_index(drop=True)

grid_df.shape

In [ ]:
weather_list = []

In [ ]:
import time

In [ ]:
for i in range(len(grid_df)):

    time.sleep(10)

    latitude = grid_df.loc[i, 'Grid_Latitude']
    longitude = grid_df.loc[i, 'Grid_Longitude']

    weather_params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': '2013-01-01',
        'end_date': '2015-12-31',
        'hourly': [
            'temperature_2m',
            'rain',
            'snowfall',
            'weather_code',
            'cloud_cover',
            'wind_speed_10m'
        ],
        'timezone': 'Europe/London'
    }

    response = requests.get(
        weather_url,
        params=weather_params,
        timeout=120
    )

    if response.status_code == 200:

        weather_data = response.json()

        temp_df = pd.DataFrame(
            weather_data['hourly']
        )

        temp_df['Grid_Latitude'] = latitude
        temp_df['Grid_Longitude'] = longitude

        weather_list.append(temp_df)

        print(i + 1, 'of', len(grid_df), 'completed')

    else:
        print(
            'Error:',
            latitude,
            longitude,
            response.status_code
        )

In [ ]:
completed_grids = []

for data in weather_list:
    latitude = data['Grid_Latitude'].iloc[0]
    longitude = data['Grid_Longitude'].iloc[0]

    completed_grids.append((latitude, longitude))

print('Completed grids:', len(completed_grids))

In [ ]:
for i in range(len(grid_df)):

    latitude = grid_df.loc[i, 'Grid_Latitude']
    longitude = grid_df.loc[i, 'Grid_Longitude']

    if (latitude, longitude) in completed_grids:
        continue

    weather_params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': '2013-01-01',
        'end_date': '2015-12-31',
        'hourly': [
            'temperature_2m',
            'rain',
            'snowfall',
            'weather_code',
            'cloud_cover',
            'wind_speed_10m'
        ],
        'timezone': 'Europe/London'
    }

    success = False

    while success == False:

        response = requests.get(
            weather_url,
            params=weather_params,
            timeout=120
        )

        if response.status_code == 200:

            weather_data = response.json()

            temp_df = pd.DataFrame(
                weather_data['hourly']
            )

            temp_df['Grid_Latitude'] = latitude
            temp_df['Grid_Longitude'] = longitude

            weather_list.append(temp_df)
            completed_grids.append((latitude, longitude))

            print(
                len(completed_grids),
                'of',
                len(grid_df),
                'completed'
            )

            success = True

            time.sleep(15)

        elif response.status_code == 429:

            print('Rate limit. Retrying this grid...')

            time.sleep(60)

        else:

            print(
                'Error:',
                latitude,
                longitude,
                response.status_code
            )

            success = True

In [ ]:
len(weather_list)

In [ ]:
weather_df = pd.concat(
    weather_list,
    ignore_index=True
)

weather_df.shape

In [ ]:
weather_df.duplicated(
    subset=[
        'Grid_Latitude',
        'Grid_Longitude',
        'time'
    ]
).sum()

In [ ]:
weather_df['time'] = pd.to_datetime(
    weather_df['time'],
    errors='coerce'
)

In [ ]:
weather_df = weather_df.rename(
    columns={'time': 'Weather_Hour'}
)

In [ ]:
weather_df.isnull().sum()

In [ ]:
print(weather_df['Weather_Hour'].min())
print(weather_df['Weather_Hour'].max())

In [ ]:
weather_df.to_csv("data/weather_london_2013_2015.csv", index=False)
print('Weather dataset saved')
print(weather_df.shape)

In [ ]:
print(accidents_df['Weather_Hour'].dtype)
print(weather_df['Weather_Hour'].dtype)

In [ ]:
print('Before Merge:', accidents_df.shape)

In [ ]:
accidents_weather_df = accidents_df.merge(
    weather_df,
    on=[
        'Grid_Latitude',
        'Grid_Longitude',
        'Weather_Hour'
    ],
    how='left'
)

In [ ]:
print('After Merge:', accidents_weather_df.shape)

In [ ]:
weather_columns = [
    'temperature_2m',
    'rain',
    'snowfall',
    'weather_code',
    'cloud_cover',
    'wind_speed_10m'
]

accidents_weather_df[weather_columns].isnull().sum()

In [ ]:
accidents_weather_df[
    [
        'Accident_Index',
        'Weather_Hour',
        'Grid_Latitude',
        'Grid_Longitude',
        'temperature_2m',
        'rain',
        'snowfall',
        'weather_code',
        'cloud_cover',
        'wind_speed_10m'
    ]
].head()

In [ ]:
len(accidents_df) == len(accidents_weather_df)

In [ ]:
test_latitude = accidents_df.loc[0, 'Grid_Latitude']
test_longitude = accidents_df.loc[0, 'Grid_Longitude']
test_date = accidents_df.loc[0, 'Date'].strftime('%Y-%m-%d')

print(test_latitude)
print(test_longitude)
print(test_date)

In [ ]:
air_quality_url = 'https://air-quality-api.open-meteo.com/v1/air-quality'

air_quality_params = {
    'latitude': test_latitude,
    'longitude': test_longitude,
    'start_date': test_date,
    'end_date': test_date,
    'hourly': [
        'pm2_5',
        'pm10',
        'nitrogen_dioxide',
        'european_aqi'
    ],
    'timezone': 'Europe/London',
    'domains': 'cams_europe'
}

air_response = requests.get(
    air_quality_url,
    params=air_quality_params,
    timeout=60
)

print(air_response.status_code)

In [ ]:
air_quality_data = air_response.json()

air_quality_test_df = pd.DataFrame(
    air_quality_data['hourly']
)

air_quality_test_df.head()

In [ ]:
air_quality_test_df.shape

In [ ]:
air_quality_test_df.columns

In [ ]:
import time

air_quality_list = []

In [ ]:
for i in range(len(grid_df)):

    latitude = grid_df.loc[i, 'Grid_Latitude']
    longitude = grid_df.loc[i, 'Grid_Longitude']

    air_quality_params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': '2013-01-01',
        'end_date': '2015-12-31',
        'hourly': [
            'pm2_5',
            'pm10',
            'nitrogen_dioxide',
            'european_aqi'
        ],
        'timezone': 'Europe/London',
        'domains': 'cams_europe'
    }

    success = False

    while success == False:

        air_response = requests.get(
            air_quality_url,
            params=air_quality_params,
            timeout=120
        )

        if air_response.status_code == 200:

            air_quality_data = air_response.json()

            temp_air_df = pd.DataFrame(
                air_quality_data['hourly']
            )

            temp_air_df['Grid_Latitude'] = latitude
            temp_air_df['Grid_Longitude'] = longitude

            air_quality_list.append(temp_air_df)

            print(
                len(air_quality_list),
                'of',
                len(grid_df),
                'completed'
            )

            success = True

            time.sleep(15)

        elif air_response.status_code == 429:

            print('Rate limit. Retrying this grid...')

            time.sleep(60)

        else:

            print(
                'Error:',
                latitude,
                longitude,
                air_response.status_code
            )

            success = True

In [ ]:
len(air_quality_list)

In [ ]:
air_quality_df = pd.concat(
    air_quality_list,
    ignore_index=True
)

air_quality_df.shape

In [ ]:
air_quality_df.duplicated(
    subset=[
        'Grid_Latitude',
        'Grid_Longitude',
        'time'
    ]
).sum()

In [ ]:
air_quality_df['time'] = pd.to_datetime(
    air_quality_df['time'],
    errors='coerce'
)

In [ ]:
air_quality_df['time'] = pd.to_datetime(
    air_quality_df['time'],
    errors='coerce'
)

In [ ]:
air_quality_df.isnull().sum()

In [ ]:
air_quality_df.columns

In [ ]:
air_quality_df['time'] = pd.to_datetime(
    air_quality_df['time'],
    errors='coerce'
)

air_quality_df.rename(
    columns={'time': 'Weather_Hour'},
    inplace=True
)

In [ ]:
air_quality_df.duplicated(
    subset=[
        'Grid_Latitude',
        'Grid_Longitude',
        'Weather_Hour'
    ]
).sum()

In [ ]:
final_df = accidents_weather_df.merge(
    air_quality_df,
    on=[
        'Grid_Latitude',
        'Grid_Longitude',
        'Weather_Hour'
    ],
    how='left'
)

In [ ]:
print('Before Merge:', accidents_weather_df.shape)
print('After Merge:', final_df.shape)

In [ ]:
air_columns = [
    'pm2_5',
    'pm10',
    'nitrogen_dioxide',
    'european_aqi'
]

final_df[air_columns].isnull().sum()

In [ ]:
len(accidents_weather_df) == len(final_df)

In [ ]:
final_df[
    [
        'Accident_Index',
        'Weather_Hour',
        'temperature_2m',
        'rain',
        'snowfall',
        'pm2_5',
        'pm10',
        'nitrogen_dioxide',
        'european_aqi'
    ]
].head()

In [ ]:
final_df.shape

In [ ]:
final_df['Accident_Index'].duplicated().sum()

In [ ]:
final_df[
    ['pm2_5', 'pm10', 'nitrogen_dioxide', 'european_aqi']
].isnull().sum()

In [ ]:
final_df.to_csv("data/final_accidents_weather_air_quality.csv", index=False)

print('Final dataset saved successfully')
print(final_df.shape)

In [ ]:
final_df[
    final_df[
        ['pm2_5', 'pm10', 'nitrogen_dioxide', 'european_aqi']
    ].isnull().any(axis=1)
][
    [
        'Accident_Index',
        'Weather_Hour',
        'Grid_Latitude',
        'Grid_Longitude',
        'pm2_5',
        'pm10',
        'nitrogen_dioxide',
        'european_aqi'
    ]
]

In [ ]:
final_df = final_df.dropna(
    subset=[
        'pm2_5',
        'pm10',
        'nitrogen_dioxide',
        'european_aqi'
    ]
).copy()

In [ ]:
final_df = final_df.reset_index(drop=True)

In [ ]:
print(final_df.shape)

final_df[
    ['pm2_5', 'pm10', 'nitrogen_dioxide', 'european_aqi']
].isnull().sum()

In [ ]:
final_df.to_csv("data/final_accidents_weather_air_quality.csv", index=False)

print('Final cleaned dataset saved')
print(final_df.shape)

In [ ]:
final_df

## General Summary

In this notebook, I completed the main data preparation part of the project. First, I loaded the `Accidents0515.csv` file from the UK road accident dataset and selected the main columns needed for the project. My goal was not to work with every column in the raw dataset, but to keep the variables that were useful for the analysis. Therefore, I kept columns related to the accident date, time, location coordinates, accident severity, road conditions, weather conditions, light conditions, number of vehicles, and number of casualties.

After that, I filtered the dataset for the years 2013, 2014, and 2015. I chose these years to keep the project size manageable and to make it easier to collect historical weather and air quality data from external APIs.

During the data cleaning stage, I checked for missing values and duplicate records. I removed rows with missing longitude, latitude, or time values because these fields were necessary for identifying the accident location and matching each accident with hourly weather and air quality data. I also checked both full duplicate rows and duplicate `Accident_Index` values, and I did not find a serious duplicate problem.

Next, I created a new target column to make accident severity easier to analyse. In the original dataset, accident severity had three categories: `Fatal`, `Serious`, and `Slight`. I converted this into a binary target variable called `Severe_Accident`. In this column, `Fatal` and `Serious` accidents were marked as `1`, while `Slight` accidents were marked as `0`. This made the target clearer and more suitable for SQL analysis, statistical analysis, and machine learning.

I also worked on the date and time columns. Using the original `Date` and `Time` columns, I created a full accident datetime column. I also created additional time-based columns such as `Year`, `Month`, `Hour`, and `Weather_Hour`. These columns will help analyse accident patterns by year, month, and hour. The `Weather_Hour` column was created specifically for merging the accident data with weather and air quality data, because the external API data was collected in hourly format.

To keep the project more focused, I limited the dataset to London. I used the `Police_Force` column and kept the codes `1` and `48`, which represent the Metropolitan Police and the City of London Police. After this step, the analysis was focused only on accidents that happened in London.

To collect weather and air quality data efficiently, I rounded the latitude and longitude values to one decimal place and created a location grid. This was an important step because sending an API request for every exact accident coordinate would have taken too much time and would not have been practical. By using the grid approach, the London accident locations were reduced to 34 main grid points, which made the API data collection process more manageable.

After that, I collected external data from Open-Meteo APIs. From the Historical Weather API, I collected variables such as temperature, rain, snowfall, cloud cover, wind speed, and weather code. From the Air Quality API, I collected PM2.5, PM10, nitrogen dioxide, and European AQI data. These datasets were collected in hourly format for the 2013–2015 period and for the London grid points created earlier.

I then merged the weather and air quality data with the accident dataset using `Grid_Latitude`, `Grid_Longitude`, and `Weather_Hour`. The weather merge was successful and did not remove any accident rows. After merging the air quality data, some missing values still remained, so those rows were removed from the final dataset.

As a result, I created a final processed dataset that combines accident information, weather conditions, and air quality indicators. This dataset contains approximately 74,000 London accident records and will be used as the main data source for the next stages of the project.

At the end of this notebook, the data collection and data cleaning stage was completed. The final dataset is now ready for SQL analysis, statistical analysis, machine learning, and dashboard development.
